In [28]:
import pandas as pd


In [29]:
df=pd.read_csv("IMDB Dataset.csv")

In [30]:
df.shape
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [31]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [32]:
df.drop_duplicates(inplace=True)

In [33]:
df.shape

(49582, 2)

In [34]:
#Pre-processing
#1.Convert to lowercase
df["review"]=df["review"].str.lower()

In [35]:
#2.Remove urls
import re

def remove_urls(text):
    text=re.sub(r"http\S+","",text)
    return text

df["review"]=df["review"].apply(remove_urls)

In [36]:
#3.Remove punctuations
def remove_punctuations(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text)
    return text

df["review"]=df["review"].apply(remove_punctuations)

In [37]:
#4.Remove HTML tags
def remove_htmltags(text):
    text=re.sub(r"<.*?>","",text)
    return text

df["review"]=df["review"].apply(remove_htmltags)

In [38]:
#5.Removing stopwords
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\RADHAGOPINATH\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\RADHAGOPINATH\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\RADHAGOPINATH\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [39]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [45]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text= text.replace(word,"")
    
    return text

df["review"]=df["review"].apply(remove_stopwords)

In [46]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [47]:
#6.Stemming
from nltk.stem import PorterStemmer

def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]

    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"]=df["review"].apply(stemming)

In [48]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [49]:
#7.Encoding
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])
y=df["sentiment"]

In [50]:
#8.Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tf=TfidfVectorizer(max_features=5000)
X=tf.fit_transform(df["review"])

In [51]:
#Dataset and Dataloader
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [57]:
import torch
from torch.utils.data import TensorDataset,DataLoader
import numpy as np 

train_set=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [58]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

In [61]:
import torch.nn as nn
import torch.optim as optim

class RNN(nn.Module):
        def __init__(self,input_size,hidden_size=128,num_layers=1):
                super().__init__()

                self.hidden_size=hidden_size
                self.num_layers=num_layers

                self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)
                self.fc=nn.Linear(hidden_size,1)

        def forward(self,x):
                # optional => shape (num of layers, batch size, hidden size)
                h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

                out, _ = self.rnn(x, h0) 
                # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
                # 2nd value = final hidden state of last timestep

                out = self.fc(out[:, -1, :])
                return out

In [62]:
input_size=X_train.shape[1]
model=RNN(input_size)
criteria=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

In [63]:
#unsqueeze squeeze
epochs=10
for epoch in range(epochs):
    model.train()
    for xb,yb in train_loader:
        optimizer.zero_grad()
        xb=xb.unsqueeze(1)
        outputs=model(xb)
        outputs=torch.sigmoid(outputs.squeeze())
        loss=criteria(outputs,yb)
        loss.backward()
        optimizer.step()

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.21190936863422394
epoch = 2/10 and loss = 0.18521854281425476
epoch = 3/10 and loss = 0.25972458720207214
epoch = 4/10 and loss = 0.2712256908416748
epoch = 5/10 and loss = 0.37605053186416626
epoch = 6/10 and loss = 0.2866269648075104
epoch = 7/10 and loss = 0.31224942207336426
epoch = 8/10 and loss = 0.15997745096683502
epoch = 9/10 and loss = 0.23776131868362427
epoch = 10/10 and loss = 0.2211567461490631


In [64]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.70132096400121
